# Getting Financial Data - Pandas Datareader

### Introduction:

This time you will get data from a website.


### Step 1. Import the necessary libraries

In [1]:
import pandas as pd
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder\
                    .appName('')\
                    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/14 18:15:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
os.getcwd()

'/tf'

In [6]:
os.chdir('/tf/pyspark_exercises/09_Time_Series/Getting_Financial_Data')

### Step 2. Create your time range (start and end variables). The start date should be 01/01/2015 and the end should today (whatever your today is).

In [ ]:
from datetime import datetime

hoy = datetime.now().date()

start_date = datetime.strptime('01/01/2015', '%m/%d/%Y').date()

In [18]:
serie_indice = pd.date_range(start=start_date,
              end=hoy)

serie_indice

DatetimeIndex(['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04',
               '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08',
               '2015-01-09', '2015-01-10',
               ...
               '2025-07-05', '2025-07-06', '2025-07-07', '2025-07-08',
               '2025-07-09', '2025-07-10', '2025-07-11', '2025-07-12',
               '2025-07-13', '2025-07-14'],
              dtype='datetime64[ns]', length=3848, freq='D')

### Step 3. Get an API key for one of the APIs that are supported by Pandas Datareader, preferably for AlphaVantage.

If you do not have an API key for any of the supported APIs, it is easiest to get one for [AlphaVantage](https://www.alphavantage.co/support/#api-key). (Note that the API key is shown directly after the signup. You do *not* receive it via e-mail.)

(For a full list of the APIs that are supported by Pandas Datareader, [see here](https://pydata.github.io/pandas-datareader/readers/index.html). As the APIs are provided by third parties, this list may change.)

In [30]:
!pip install dotenv



[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [33]:
from dotenv import load_dotenv

load_dotenv()
CLAVE = 'API_KEY'
API_KEY = os.getenv(CLAVE)



### Step 4. Use Pandas Datarader to read the daily time series for the Apple stock (ticker symbol AAPL) between 01/01/2015 and today, assign it to df_apple and print it.

In [35]:
!pip install pandas_datareader


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 30.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [44]:
import pandas_datareader.data as web


start_date = datetime.datetime(2015, 1, 1)
end_date = datetime.datetime.today()

# Descargar datos usando Alpha Vantage
df_apple = web.DataReader('AAPL', 'av-daily', start=start_date, end=end_date, api_key=API_KEY)

# Mostrar resultados
print(df_apple.head())

              open    high      low   close    volume
2015-01-02  111.39  111.44  107.350  109.33  53204626
2015-01-05  108.29  108.65  105.410  106.25  64285491
2015-01-06  106.54  107.43  104.630  106.26  65797116
2015-01-07  107.20  108.20  106.695  107.75  40105934
2015-01-08  109.23  112.15  108.700  111.89  59364547


In [45]:
apple = spark.createDataFrame(df_apple)
apple.show()

+-------+--------+-------+------+---------+
|   open|    high|    low| close|   volume|
+-------+--------+-------+------+---------+
| 111.39|  111.44| 107.35|109.33| 53204626|
| 108.29|  108.65| 105.41|106.25| 64285491|
| 106.54|  107.43| 104.63|106.26| 65797116|
|  107.2|   108.2|106.695|107.75| 40105934|
| 109.23|  112.15|  108.7|111.89| 59364547|
| 112.67|  113.25| 110.21|112.01| 53315099|
|  112.6|  112.63|  108.8|109.25| 49650790|
| 111.43|   112.8| 108.91|110.22| 67091928|
| 109.04|  110.49|  108.5| 109.8| 48956588|
|  110.0|  110.06| 106.66|106.82| 60013996|
| 107.03|  107.58|  105.2|105.99| 78513345|
| 107.84|108.9667|  106.5|108.72| 49899907|
| 108.95|  111.06| 108.27|109.55| 48575897|
| 110.26|  112.47| 109.72| 112.4| 53796409|
|  112.3|  113.75| 111.53|112.98| 46464828|
| 113.74|114.3626|  112.8| 113.1| 55614979|
| 112.42|  112.48| 109.03|109.14| 95568749|
|117.625|  118.12| 115.31|115.31|146477063|
| 116.32|  119.19| 115.56| 118.9| 84436432|
|  118.4|   120.0| 116.85|117.16

### Step 5. Add a new column "stock" to the dataframe and add the ticker symbol

In [50]:
apple = apple.withColumn('stock', F.lit('AAPL'))
apple.show()

+-------+--------+-------+------+---------+-----+
|   open|    high|    low| close|   volume|stock|
+-------+--------+-------+------+---------+-----+
| 111.39|  111.44| 107.35|109.33| 53204626| AAPL|
| 108.29|  108.65| 105.41|106.25| 64285491| AAPL|
| 106.54|  107.43| 104.63|106.26| 65797116| AAPL|
|  107.2|   108.2|106.695|107.75| 40105934| AAPL|
| 109.23|  112.15|  108.7|111.89| 59364547| AAPL|
| 112.67|  113.25| 110.21|112.01| 53315099| AAPL|
|  112.6|  112.63|  108.8|109.25| 49650790| AAPL|
| 111.43|   112.8| 108.91|110.22| 67091928| AAPL|
| 109.04|  110.49|  108.5| 109.8| 48956588| AAPL|
|  110.0|  110.06| 106.66|106.82| 60013996| AAPL|
| 107.03|  107.58|  105.2|105.99| 78513345| AAPL|
| 107.84|108.9667|  106.5|108.72| 49899907| AAPL|
| 108.95|  111.06| 108.27|109.55| 48575897| AAPL|
| 110.26|  112.47| 109.72| 112.4| 53796409| AAPL|
|  112.3|  113.75| 111.53|112.98| 46464828| AAPL|
| 113.74|114.3626|  112.8| 113.1| 55614979| AAPL|
| 112.42|  112.48| 109.03|109.14| 95568749| AAPL|


In [46]:
df_apple['stock'] = 'AAPL'

In [47]:
df_apple

,open,high,low,close,volume,stock
2015-01-02,111.390,111.44,107.350,109.33,53204626,AAPL
2015-01-05,108.290,108.65,105.410,106.25,64285491,AAPL
2015-01-06,106.540,107.43,104.630,106.26,65797116,AAPL
2015-01-07,107.200,108.20,106.695,107.75,40105934,AAPL
2015-01-08,109.230,112.15,108.700,111.89,59364547,AAPL
...,...,...,...,...,...,...
2025-07-07,212.680,216.23,208.800,209.95,50228984,AAPL
2025-07-08,210.100,211.43,208.450,210.01,42848928,AAPL
2025-07-09,209.530,211.33,207.220,211.14,48749367,AAPL
2025-07-10,210.505,213.48,210.030,212.41,44443635,AAPL


### Step 6. Repeat the two previous steps for a few other stocks, always creating a new dataframe: Tesla, IBM and Microsoft. (Ticker symbols TSLA, IBM and MSFT.)

In [ ]:

nuevos_datos = ['TSLA', 'IBM', 'MSFT']


def create_data(lista):
    dfs = []
    for dato in lista:
        df = web.DataReader(dato, 'av-daily', start_date, end_date, api_key=API_KEY)
        df['stock'] = dato
        dfs.append(df)
    
    return pd.concat(dfs)
    

df = create_data(nuevos_datos)

In [57]:
df

,open,high,low,close,volume,stock
2015-01-02,222.87,223.2500,213.2600,219.310,4764443,TSLA
2015-01-05,214.55,216.5000,207.1626,210.090,5368477,TSLA
2015-01-06,210.06,214.2000,204.2100,211.280,6261936,TSLA
2015-01-07,213.35,214.7800,209.7800,210.950,2968390,TSLA
2015-01-08,212.81,213.7999,210.0100,210.615,3442509,TSLA
...,...,...,...,...,...,...
2025-07-07,497.38,498.7500,495.2250,497.720,13981605,MSFT
2025-07-08,497.24,498.2000,494.1100,496.620,11846586,MSFT
2025-07-09,500.30,506.7800,499.7400,503.510,18659538,MSFT
2025-07-10,503.05,504.4400,497.7500,501.480,16498740,MSFT


In [58]:
df_rest = spark.createDataFrame(df)

df_rest.show()

+------+--------+--------+-------+--------+-----+
|  open|    high|     low|  close|  volume|stock|
+------+--------+--------+-------+--------+-----+
|222.87|  223.25|  213.26| 219.31| 4764443| TSLA|
|214.55|   216.5|207.1626| 210.09| 5368477| TSLA|
|210.06|   214.2|  204.21| 211.28| 6261936| TSLA|
|213.35|  214.78|  209.78| 210.95| 2968390| TSLA|
|212.81|213.7999|  210.01|210.615| 3442509| TSLA|
|208.92|  209.98|  204.96| 206.66| 4580722| TSLA|
|203.05|  204.47|  199.25| 202.21| 5950280| TSLA|
|203.32|  207.61| 200.911| 204.25| 4477320| TSLA|
|185.83|   195.2|   185.0| 192.69|11551855| TSLA|
|194.49|195.7499|   190.0| 191.87| 5216524| TSLA|
| 190.7|  194.49|  189.65| 193.07| 3603158| TSLA|
|193.87|194.1199|  187.04| 191.93| 4503182| TSLA|
|189.55|  198.68|  189.51| 196.57| 4153043| TSLA|
| 197.0|  203.24|   195.2| 201.62| 4116905| TSLA|
|200.29|   203.5|  198.33| 201.29| 3442371| TSLA|
|201.83|  208.62|  201.05| 206.55| 3234522| TSLA|
|204.42|  208.03|   203.3| 205.98| 2781024| TSLA|


### Step 7. Combine the four separate dataFrames into one combined dataFrame df that holds the information for all four stocks

In [62]:
df_total = apple.union(df_rest)


In [67]:
df_total.tail(10)

[Row(open=497.55, high=499.3, low=493.03, close=495.94, volume=34539236, stock='MSFT'),
 Row(open=497.04, high=500.76, low=495.33, close=497.41, volume=28368991, stock='MSFT'),
 Row(open=496.47, high=498.05, low=490.98, close=492.05, volume=19945375, stock='MSFT'),
 Row(open=489.99, high=493.5, low=488.7, close=491.09, volume=16319641, stock='MSFT'),
 Row(open=493.81, high=500.13, low=493.44, close=498.84, volume=13984829, stock='MSFT'),
 Row(open=497.38, high=498.75, low=495.225, close=497.72, volume=13981605, stock='MSFT'),
 Row(open=497.24, high=498.2, low=494.11, close=496.62, volume=11846586, stock='MSFT'),
 Row(open=500.3, high=506.78, low=499.74, close=503.51, volume=18659538, stock='MSFT'),
 Row(open=503.05, high=504.44, low=497.75, close=501.48, volume=16498740, stock='MSFT'),
 Row(open=498.47, high=505.03, low=497.795, close=503.32, volume=16459512, stock='MSFT')]

In [60]:
df_all = pd.concat([df_apple,
                     df])

df_all

,open,high,low,close,volume,stock
2015-01-02,111.39,111.44,107.350,109.33,53204626,AAPL
2015-01-05,108.29,108.65,105.410,106.25,64285491,AAPL
2015-01-06,106.54,107.43,104.630,106.26,65797116,AAPL
2015-01-07,107.20,108.20,106.695,107.75,40105934,AAPL
2015-01-08,109.23,112.15,108.700,111.89,59364547,AAPL
...,...,...,...,...,...,...
2025-07-07,497.38,498.75,495.225,497.72,13981605,MSFT
2025-07-08,497.24,498.20,494.110,496.62,11846586,MSFT
2025-07-09,500.30,506.78,499.740,503.51,18659538,MSFT
2025-07-10,503.05,504.44,497.750,501.48,16498740,MSFT


### Step 8. Shift the stock column into the index (making it a multi-level index consisting of the ticker symbol and the date).

### Step 7. Create a dataFrame called vol, with the volume values.

### Step 8. Aggregate the data of volume to weekly.
Hint: Be careful to not sum data from the same week of 2015 and other years.

### Step 9. Find all the volume traded in the year of 2015